In [ ]:
import os
from os import path
import pickle
import numpy as np
from tqdm import tqdm

from macaquethings.plotting.default_styles import *
from macaquethings.plotting.util import legend_without_duplicate_labels

import matplotlib.pyplot as plt
import seaborn as sns

figure_style()  # set consistent plotting defaults for all figs
%matplotlib widget

# Load Decoding Performance

In [ ]:
# specify general options
monkey = 'monkeyF'  # use for the whole script

array_selections = list(np.arange(1, 17))  # all arrays

# array 6 is broken for monkeyN
if monkey == "monkeyN":
    array_selections.remove(6)  # 6 is reported broken
else:
    array_selections.remove(7)  # no good channels in 7

    
sessions = "0_3_4_5" if monkey == "monkeyN" else "0_1_2_3_4_5"  # all sessions

if monkey == 'monkeyF':
    array_to_roi = {
        1: 'v1',
        2: 'v1',
        3: 'v1',
        4: 'v1',
        5: 'v1',
        6: 'v1',
        7: 'v1',
        8: 'v1',
        9: 'it',
        10: 'it',
        11: 'it',
        12: 'it',
        13: 'it',
        14: 'v4',
        15: 'v4',
        16: 'v4',
    }
else:
    array_to_roi = {
        1: 'v1',
        2: 'v1',
        3: 'v1',
        4: 'v1',
        5: 'v1',
        6: 'v1',
        7: 'v1',
        8: 'v1',
        9: 'v4',
        10: 'v4',
        11: 'v4',
        12: 'v4',
        13: 'it',
        14: 'it',
        15: 'it',
        16: 'it',
    }    

roi_to_color = {
    'v1': 'tab:blue',
    'v4': 'tab:orange',
    'it': 'tab:green'
}

array_to_color = {}

n_v1 = (np.array(list(array_to_roi.values())) == 'v1').sum()
n_v4 = (np.array(list(array_to_roi.values())) == 'v4').sum()
n_it = (np.array(list(array_to_roi.values())) == 'it').sum()

blues = sns.color_palette('Blues', n_v1 + 1)[1:]
oranges = sns.color_palette('Oranges', n_v4 + 1)[1:]
greens = sns.color_palette('Greens', n_it + 1)[1:]

cmaps_per_roi = {
    'v1': blues,
    'v4': oranges,
    'it': greens
}

v1_counter = 0
v4_counter = 0
it_counter = 0

for arr in array_to_roi.keys():
    roi = array_to_roi[arr]
    cmap = cmaps_per_roi[roi]
    if roi == 'v1':
        color = cmap[v1_counter]
        v1_counter += 1
    elif roi == 'v4':
        color = cmap[v4_counter]
        v4_counter += 1
    else:
        color = cmap[it_counter]
        it_counter += 1
    array_to_color[arr] = color


from matplotlib.lines import Line2D
import matplotlib.pyplot as plt

def create_legend(array_to_color, linestyles):
    """
    Create a custom legend:
    - One entry per array color
    - One entry per linestyle (black)
    
    Parameters:
        array_to_color : dict
            Mapping of array index -> color
        linestyles : dict
            Mapping of label type -> linestyle (e.g., {'label1':'-', 'label2':'--'})
    """
    legend_elements = []

    # One entry per array (color)
    for arr, color in array_to_color.items():
        legend_elements.append(Line2D([0], [0], color=color, lw=2, label=f'Array {arr}'))

    # One entry per linestyle (black)
    for label_type, linestyle in linestyles.items():
        legend_elements.append(Line2D([0], [0], color='black', lw=2, linestyle=linestyle, label=label_type))

    # Create figure for the legend only
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.legend(handles=legend_elements, loc='center', ncol=4, frameon=False)
    ax.axis('off')  # hide axes
    plt.show()

create_legend(array_to_color, linestyles={'category': '-', 'stimulus identity': '--'})
plt.savefig(path.join('decoding_performance_panels', f'{monkey}_full_array_legend.svg'))

In [ ]:

decoding_dir = path.join('..', '..', 'results', 'decoding', f'{monkey}_stratifyimages_allsess')
array_decoding_pkls = []

arr_id_info = []
label_info = []
roi_info = []

for array in array_selections:
    roi = '1_2_3'
    arrays = str(array)
    fname_cat = f"{monkey}-labels_category-sessions_{sessions}-rois_{roi}-arrays_{arrays}-baseline_0-standardize_1-lfp.pkl"
    fname_file = f"{monkey}-labels_filenames-sessions_{sessions}-rois_{roi}-arrays_{arrays}-baseline_0-standardize_1-lfp.pkl"
    array_decoding_pkls.append(fname_cat)
    array_decoding_pkls.append(fname_file)
    arr_id_info.append(array)
    arr_id_info.append(array)
    label_info.append("category")
    label_info.append("stimulus identity")
    roi_info.append(array_to_roi[array])
    roi_info.append(array_to_roi[array])


In [ ]:
# load
print()
print("loading array decoding results:")
array_decoding_results = []
for decoding_pkl in tqdm(array_decoding_pkls):
    with open(path.join(decoding_dir, decoding_pkl), 'rb') as f:
        res = pickle.load(f)
        t = res['times'].squeeze()
        acc = res['accuracies']
        array_decoding_results.append({
            'time': t,
            'accuracy': acc
        })


In [ ]:
# FIG 1 --- all arrays
fig_decoding_per_array = plt.figure(figsize=HALF_PANEL_SIZE)

for arr_id, label, res in zip(arr_id_info, label_info, array_decoding_results):
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
plt.xlabel('time (ms)')
plt.ylabel('decoding accuracy')
plt.title(f"Decoding accuracy per array {monkey}")
plt.savefig(path.join('decoding_performance_panels', f'{monkey}_per_array_all.svg'))

# Plot for each ROI

In [ ]:
xlim = (-30, 400)


# FIG 2 --- V1
fig_decoding_per_array = plt.figure(figsize=QUARTER_PANEL_SIZE)

for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not roi == 'v1':
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
plt.xlabel('time (ms)')
plt.ylabel('decoding accuracy')
plt.title(f"V1 - {monkey}")
plt.xlim(xlim)
plt.savefig(path.join('decoding_performance_panels', f'{monkey}_per_array_v1.svg'))

# FIG 2 --- V4
fig_decoding_per_array = plt.figure(figsize=QUARTER_PANEL_SIZE)

for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not roi == 'v4':
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
plt.xlabel('time (ms)')
plt.ylabel('decoding accuracy')
plt.title(f"V4 - {monkey}")
plt.xlim(xlim)
plt.savefig(path.join('decoding_performance_panels', f'{monkey}_per_array_v4.svg'))

# FIG 2 --- IT
fig_decoding_per_array = plt.figure(figsize=QUARTER_PANEL_SIZE)

for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not roi == 'it':
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
plt.xlabel('time (ms)')
plt.ylabel('decoding accuracy')
plt.title(f"IT - {monkey}")
plt.xlim(xlim)
plt.savefig(path.join('decoding_performance_panels', f'{monkey}_per_array_it.svg'))



# Plot for each ROI - category only

In [ ]:
xlim = (-30, 400)
panelsize = THIRD_PANEL_SIZE

# FIG 3 --- V1
fig_decoding_per_array = plt.figure(figsize=panelsize)

for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not (roi == 'v1' and label == 'category'):
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
plt.xlabel('time (ms)')
plt.ylabel('decoding accuracy')
plt.title(f"V1 - {monkey}")
plt.xlim(xlim)
plt.savefig(path.join('decoding_performance_panels', f'{monkey}_category_per_array_v1.svg'))

# FIG 3 --- V4
fig_decoding_per_array = plt.figure(figsize=panelsize)

for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not (roi == 'v4' and label == 'category'):
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
plt.xlabel('time (ms)')
plt.ylabel('decoding accuracy')
plt.title(f"V4 - {monkey}")
plt.xlim(xlim)
plt.savefig(path.join('decoding_performance_panels', f'{monkey}_category_per_array_v4.svg'))

# FIG 3 --- IT
fig_decoding_per_array = plt.figure(figsize=panelsize)

for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not (roi == 'it' and label == 'category'):
        
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
plt.xlabel('time (ms)')
plt.ylabel('decoding accuracy')
plt.title(f"IT - {monkey}")
plt.xlim(xlim)
plt.savefig(path.join('decoding_performance_panels', f'{monkey}_category_per_array_it.svg'))



# Category per Array - all ROIs in one Panel

In [ ]:
xlim = (-30, 400)
panelsize = HALF_PANEL_SIZE 

fig_decoding_per_array = plt.figure(figsize=panelsize)
# --- V1

for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not (roi == 'v1' and label == 'category'):
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
plt.xlabel('time (ms)')
plt.ylabel('decoding accuracy')
plt.title(f"category - {monkey}")
plt.xlim(xlim)

# --- V4
for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not (roi == 'v4' and label == 'category'):
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')

# --- IT
for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
    if not (roi == 'it' and label == 'category'):
        
        continue
    color = array_to_color[arr_id]
    plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')

plt.savefig(path.join('decoding_performance_panels', f'{monkey}_category_per_array_all_rois.svg'))



# Plot each array separately

In [ ]:
# FIG 2 --- IT

for arr in np.unique(arr_id_info):
    fig_decoding_single_arr = plt.figure(figsize=QUARTER_PANEL_SIZE)
    for arr_id, label, roi, res in zip(arr_id_info, label_info, roi_info, array_decoding_results):
        if not arr_id == arr:
            continue
        color = 'k'
        plt.plot(res['time'], res['accuracy'], label=str(arr_id), color=color, linestyle='-' if label == 'category' else '--')
    plt.xlabel('time (ms)')
    plt.ylabel('decoding accuracy')
    plt.title(f"Array {arr} - {monkey}")
    plt.xlim(xlim)
    plt.savefig(path.join('decoding_performance_panels', f'{monkey}_array_{arr}.svg'))
    
